In [1]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.optimize import curve_fit
from scipy.stats import landau, norm, trim_mean, trimboth

import warnings
warnings.filterwarnings('ignore')

In [2]:
def get_data_det(file_name, num1, num2):
    file = ROOT.TFile.Open(file_name)
    detdir = file.Get("VirtualDetector")

    detname = "det_" + str(num1) + "." + str(num2)

    # print(detname)
    
    df_det = ROOT.RDataFrame(detname, detdir)

    X = df_det.AsNumpy(columns=["x"])["x"]
    Y = df_det.AsNumpy(columns=["y"])["y"]
    Z = df_det.AsNumpy(columns=["z"])["z"]
    PX = df_det.AsNumpy(columns=["Px"])["Px"]
    PY = df_det.AsNumpy(columns=["Py"])["Py"]
    PZ = df_det.AsNumpy(columns=["Pz"])["Pz"]
    T = df_det.AsNumpy(columns=["t"])["t"]
    eventID = df_det.AsNumpy(columns=["EventID"])["EventID"]
    parentID = df_det.AsNumpy(columns=["ParentID"])["ParentID"]
    

    x = X[parentID == 0]
    y = Y[parentID == 0]
    z = Z[parentID == 0]
    px = PX[parentID == 0]
    py = PY[parentID == 0]
    pz = PZ[parentID == 0]
    t = T[parentID == 0]
    event = eventID[parentID == 0]

    return [x, y, z, px, py, pz, t, event]

def get_data_trace(filename, trackname):
    file = ROOT.TFile.Open(filename)
    detdir = file.Get("Trace")

    detname = str(trackname)

    # print(detname)
    
    df_det = ROOT.RDataFrame(detname, detdir)

    X = df_det.AsNumpy(columns=["x"])["x"]
    Y = df_det.AsNumpy(columns=["y"])["y"]
    Z = df_det.AsNumpy(columns=["z"])["z"]
    PX = df_det.AsNumpy(columns=["Px"])["Px"]
    PY = df_det.AsNumpy(columns=["Py"])["Py"]
    PZ = df_det.AsNumpy(columns=["Pz"])["Pz"]
    T = df_det.AsNumpy(columns=["t"])["t"]
    parentID = df_det.AsNumpy(columns=["ParentID"])["ParentID"]
    Pathlength = df_det.AsNumpy(columns=["PathLength"])["PathLength"]
    

    x = X[parentID == 0]
    y = Y[parentID == 0]
    z = Z[parentID == 0]
    px = PX[parentID == 0]
    py = PY[parentID == 0]
    pz = PZ[parentID == 0]
    t = T[parentID == 0]
    pathlength = Pathlength[parentID == 0]

    return [x, y, z, px, py, pz, t, pathlength]

In [ ]:
m_mu = 105.7 # MeV/c^2
c = 299.792 # mm/ns
filename = "det_in_gap/1e6_bunch_3layers/g4beamline.root"

n_layers = 3
cell_list = []
x_list  = []
y_list  = []
z_list  = []
px_list = []
py_list = []
pz_list = []
t_list  = []
event_list = []
vals_temp = []
for cell in np.arange(0,61):
    
    if(cell%5==2 or cell%5==4):
        idx = int(np.floor((cell%5)/4)+2*np.floor(cell/5))
        print(f'cell {cell}, index {idx}' )
        cell_list.append(cell)
        x_list.append([])
        y_list.append([])
        z_list.append([])
        px_list.append([])
        py_list.append([])
        pz_list.append([])
        t_list.append([])
        event_list.append([])
        
        for i in np.arange(0,2*n_layers):
                vals_temp = get_data_det(filename,cell,i)
                x_list[idx].append(vals_temp[0])
                y_list[idx].append(vals_temp[1])
                z_list[idx].append(vals_temp[2])
                px_list[idx].append(vals_temp[3])
                py_list[idx].append(vals_temp[4])
                pz_list[idx].append(vals_temp[5])
                t_list[idx].append(vals_temp[6])
                event_list[idx].append(vals_temp[7])

cell 2, index 0
cell 4, index 1
cell 7, index 2
cell 9, index 3
cell 12, index 4
cell 14, index 5


In [ ]:
n_traces = 10000
filename_trace = "traces/g4beamline.root"
trace_x_list = []
trace_y_list = []
trace_z_list = []
trace_t_list = []
trace_pl_list = []
for k in np.arange(1,n_traces):
    if (k%1000==0): print(k)
    vals_temp = get_data_trace(filename_trace,"Ev"+str(k)+"Trk0")
    # vals_temp = get_data_trace("12layers_bunch/g4beamline.root","Ev"+str(k)+"Trk0")
    trace_x_list.append(vals_temp[0])
    trace_y_list.append(vals_temp[1])
    trace_z_list.append(vals_temp[2])
    trace_t_list.append(vals_temp[6])
    trace_pl_list.append(vals_temp[7])
    
vals_temp = get_data_trace(filename_trace,"ReferenceParticle")
# vals_temp = get_data_trace("12layers_bunch/g4beamline.root","ReferenceParticle")
trace_x_list.append(vals_temp[0])
trace_y_list.append(vals_temp[1])
trace_z_list.append(vals_temp[2])
trace_t_list.append(vals_temp[6])
trace_pl_list.append(vals_temp[7])

trace_arrays = []

for i in np.arange(0,n_traces):
    trace_arrays.append([])
    trace_arrays[i].append(trace_x_list[i])
    trace_arrays[i].append(trace_y_list[i])
    trace_arrays[i].append(trace_z_list[i])
    trace_arrays[i].append(trace_t_list[i])
    trace_arrays[i].append(trace_pl_list[i])

trace_arrays_np = []

for tr in trace_arrays:
    arr = np.asarray(tr)  # shape (5, N)
    trace_arrays_np.append(arr)

def track_length_path(track, z1, z2):
    x, y, z, t, pl = track

    # sort
    order = np.argsort(t)
    z = z[order]
    pl = pl[order]

    # handle edge cases
    if len(z) < 2:
        return 0

    # interpolate PathLength at boundaries
    L1 = np.interp(z1, z, pl)
    L2 = np.interp(z2, z, pl)

    return L2 - L1

z_det_avgs = []
for idx in np.arange(0,24):
    mean_z_A = np.mean(np.concatenate(z_list[idx][:n_layers]))
    mean_z_B = np.mean(np.concatenate(z_list[idx][n_layers:]))
    z_det_avgs.append([mean_z_A,mean_z_B])
    # print(f"z detector group positions in cell {cell}: {mean_z_A:.2f} to {mean_z_B:.2f}")

cell_trlen_avgs_path = []
cell_trlen_stdev = []
cell_trlen_norm_avgs = []


for idx in np.arange(0,24):
    cell_lens_path = []
    for trace_num in np.arange(0,n_traces):

        if (np.max(trace_z_list[trace_num]) >= z_det_avgs[idx][1]) and (np.min(trace_z_list[trace_num]) <= z_det_avgs[idx][0]):
            
            track_len_path = track_length_path(trace_arrays_np[trace_num],z_det_avgs[idx][0],z_det_avgs[idx][1])

            if(track_len_path != 0):
                cell_lens_path.append(track_len_path)
            
    trim_frac = 0.01
    path_trim = trimboth(cell_lens_path,trim_frac)

    
    cell_trlen_avgs_path.append(np.mean(path_trim))
    cell_trlen_stdev.append(np.std(path_trim))
    cell_trlen_norm_avgs.append(np.mean(cell_lens_path)/(z_det_avgs[idx][1] - z_det_avgs[idx][0]))
         

In [ ]:
idx = 1
z_det_avgs[idx][1] - z_det_avgs[idx][0]

In [ ]:
def gaussian(x, mu, sigma, A):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def get_gaussian_params(t_vals, num_t_bins):
    t_mean = np.mean(t_vals)
    t_spread = 5*np.std(t_vals)

    lower_bounds_gauss = [1, 0.001, 1.0]
    upper_bounds_gauss = [1000, 20.0, 4000000.0]
      # lower_bounds_gauss = [0, 0.001, 1.0]
        # upper_bounds_gauss = [100000, 10.0, 40000.0]

    hist_counts_func, bin_edges = np.histogram(
        t_vals,
        range=(t_mean - t_spread, t_mean + t_spread),
        bins=num_t_bins
    )

    bin_centers_func = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    mu_init = bin_centers_func[np.argmax(hist_counts_func)]
    sigma_init = np.std(bin_centers_func)
    A_init = np.max(hist_counts_func)

    # print(f'initial guess: {[mu_init, sigma_init, A_init]}')

    params_gauss, covariance_gauss = curve_fit(
        gaussian,
        bin_centers_func,
        hist_counts_func,
        p0=[mu_init, sigma_init, A_init],
        bounds=(lower_bounds_gauss,upper_bounds_gauss)
    )

    mu, sigma, A = params_gauss

    mu_unc = np.nan
    try:
        mu_unc = np.sqrt(np.diag(covariance_gauss))[0]
    except ValueError:
        pass

    return mu, sigma, A, mu_unc, bin_centers_func, hist_counts_func

def digitize_return_only_single_hits(pixel_size_in,x_in,y_in,t_in):
    
    size_x = 81.5
    # binsize_x = pixel_sizesize # mm
    bins_x = int(2*size_x / pixel_size_in)
    
    x_edges = np.linspace(-size_x, size_x, bins_x + 1)
    y_edges = np.linspace(-size_x, size_x, bins_x + 1)
    
    # digitize gives bin index (1..bins), subtract 1 → (0..bins-1)
    ix = np.digitize(x_in, x_edges) - 1
    iy = np.digitize(y_in, y_edges) - 1

    t_in = np.asarray(t_in)
    
    valid = (ix >= 0) & (ix < bins_x) & (iy >= 0) & (iy < bins_x)
    ix, iy, t_in = ix[valid], iy[valid], t_in[valid]
    
    pixel_id = ix * bins_x + iy

    order = np.argsort(pixel_id)
    pixel_id_sorted = pixel_id[order]
    t_sorted = t_in[order]
    
    # Get unique pixel IDs along with counts and the index of first occurrence
    unique_ids, first_indices, counts = np.unique(
        pixel_id_sorted, return_index=True, return_counts=True
    )
    
    # Keep only pixels that were hit exactly once
    single_hit_mask = counts == 1
    single_hit_indices = first_indices[single_hit_mask]
    
    t_single_hits = t_sorted[single_hit_indices]
    
    return t_single_hits

def smeared(t_in,resolution):

    t_in_smeared = np.random.normal(t_in,resolution,len(t_in))
    
    return t_in_smeared

In [ ]:
def get_time_at_threshold(t_in,pc_threshold,n_clocks_in,clock_spacing_in,add):

    add_to_last_bin = add

    middle = np.mean(t_in)


    min_t = middle - (n_clocks_in/2) * clock_spacing_in
    regular_max_t = min_t + (n_clocks_in-1) * clock_spacing_in

    overflow_max_t = regular_max_t + clock_spacing_in + add_to_last_bin

    my_edges = np.concatenate([np.arange(min_t,regular_max_t + clock_spacing_in,clock_spacing_in),[overflow_max_t]])
    counts, edges = np.histogram(t_in, bins=my_edges)


    # Add all t < min_t to the first bin
    counts[0] += np.sum(t_in < min_t)

    # Plot cumulative histogram
    cumulative = np.cumsum(counts)

    
    target = pc_threshold*cumulative[-1]

    index, value = min(
            enumerate(cumulative),
            key=lambda item: abs(item[1] - target)
        )

    timing_at_threshold = min_t + (index+0.5)*clock_spacing_in

    return timing_at_threshold


def get_time_at_threshold_approxmid(t_in,pc_threshold,n_clocks_in,clock_spacing_in,add,precision):

    add_to_last_bin = add

    middle = np.round(np.mean(t_in),precision)


    min_t = middle - (n_clocks_in/2) * clock_spacing_in
    regular_max_t = min_t + (n_clocks_in-1) * clock_spacing_in

    overflow_max_t = regular_max_t + clock_spacing_in + add_to_last_bin

    my_edges = np.concatenate([np.arange(min_t,regular_max_t + clock_spacing_in,clock_spacing_in),[overflow_max_t]])
    counts, edges = np.histogram(t_in, bins=my_edges)


    # Add all t < min_t to the first bin
    counts[0] += np.sum(t_in < min_t)

    # Plot cumulative histogram
    cumulative = np.cumsum(counts)

    
    target = pc_threshold*cumulative[-1]

    index, value = min(
            enumerate(cumulative),
            key=lambda item: abs(item[1] - target)
        )

    timing_at_threshold = min_t + (index+0.5)*clock_spacing_in

    return timing_at_threshold


    

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
plt.rcParams['grid.alpha'] = 0.5


res = 0.03
idx = 12
det = 0
pixel_size = 0.025
threshold = 0.5

t_first = digitize_return_only_single_hits(pixel_size,x_list[idx][det],y_list[idx][det],t_list[idx][det])

t = smeared(t_first,res)
middle = np.round(np.mean(t), decimals=2)


for i in range(4):

    n_clocks = 32*(2**i)
    clock_spacing = 0.006 # ps
    
    # min_t = min(t)
    # max_t = min(t) + n_clocks*clock_spacing

    middle = np.round(np.mean(t), decimals=1)

    min_t = middle - (n_clocks/2)*clock_spacing
    regular_max_t = min_t + (n_clocks-1)*clock_spacing

    overflow_max_t = regular_max_t + clock_spacing + 0.5

    my_edges = np.concatenate([np.arange(min_t,regular_max_t + clock_spacing,clock_spacing),[overflow_max_t]])
    # print(my_edges)
    counts, edges = np.histogram(t, bins=my_edges)
    # print(edges)

    # min_t = np.floor(min_t / clock_spacing) * clock_spacing
    # max_t = np.floor(max_t / clock_spacing) * clock_spacing
    
    # nbins = int((overflow_max_t-min_t)/clock_spacing)
    
    # ax.hist(t, bins=nbins, range=(min_t,max_t), cumulative=True, zorder=10-i, label=f'{n_clocks} clocks')
    # counts, edges = np.histogram(
    #     t,
    #     bins=nbins,
    #     range=(min_t, ovmax_t)
    # )

    # Add all t < min_t to the first bin
    counts[0] += np.sum(t < min_t)

    # Plot cumulative histogram
    cumulative = np.cumsum(counts)

    # Draw actual filled histogram bars
    widths = np.diff(edges)

    ax.bar(
        edges[:-1],
        cumulative,
        width=widths,
        align='edge',
        zorder=10-i,
        label=f'{n_clocks} clocks'
    )
    timing = get_time_at_threshold(t_first,threshold,n_clocks,clock_spacing,0.5)
    target = 0.5*cumulative[-1]
    
    ax.axvline(timing,color='black',zorder=1000)
    print(timing)
    
fs = 18

ax.grid(alpha=0.5)

ax.set_ylabel('number of hits')
ax.set_xlabel('time [ns]')
ax.legend()

# plt.xticks(np.arange(min_t, max_t, res))
# ax.tick_params(axis='x', which='major', labelsize=5)

# plt.set_xlim(min_t, max_t)
plt.title(f'Timing histogram, {(clock_spacing*1000):.0f}ps bins, clocked from first hit\ngap {idx}, sensor {det}')
plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
plt.rcParams['grid.alpha'] = 0.5


res = 0.03
pixel_size = 0.025
idx = 14
det = 0
clock_spacing = 0.003 # ps

t_first = digitize_return_only_single_hits(pixel_size,x_list[idx][det],y_list[idx][det],t_list[idx][det])
t_all = t_list[idx][det]


t = smeared(t_first,res)
middle = np.round(np.mean(t), decimals=2)



for i in range(4):

    t = smeared(t_all,res)
    middle = np.round(np.mean(t), decimals=1)

    n_clocks = 32*(2**i)
    
    
    add_to_end = 1

    middle = np.round(np.mean(t), decimals=1)

    min_t = middle - (n_clocks/2)*clock_spacing
    regular_max_t = min_t + (n_clocks-1)*clock_spacing

    overflow_max_t = regular_max_t + clock_spacing + add_to_end

    my_edges = np.concatenate([np.arange(min_t,regular_max_t + clock_spacing,clock_spacing),[overflow_max_t]])
    counts, edges = np.histogram(t, bins=my_edges)


    # Add all t < min_t to the first bin
    counts[0] += np.sum(t < min_t)

    # Plot cumulative histogram
    cumulative = np.cumsum(counts)

    # Draw actual filled histogram bars
    widths = np.diff(edges)

    ax.bar(
        edges[:-1],
        cumulative,
        width=widths,
        align='edge',
        zorder=10-i,
        label=f'{n_clocks} clocks: all hits'
    )

    timing = get_time_at_threshold(t,threshold,n_clocks,clock_spacing,add_to_end)
    target = 0.5*cumulative[-1]
    print(target,cumulative[-1])
    if(i==3):
        ax.axvline(timing,color='black',zorder=13,label=f'timing: all hits ({timing:.4f}ns)')
        ax.axhline(target,color='black',zorder=13,label=f'threshold ({threshold*100:.0f}%): all hits')


fs = 18

ax.grid(alpha=0.5)

ax.set_ylabel('number of hits')
ax.set_xlabel('time [ns]')
ax.legend()

# plt.xticks(np.arange(min_t, max_t, res))
# ax.tick_params(axis='x', which='major', labelsize=5)

# plt.set_xlim(min_t, max_t)
plt.title(f'Timing histogram, {(clock_spacing*1000):.0f}ps bins, clocked from first hit\ngap {idx}, sensor {det}')
plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
plt.rcParams['grid.alpha'] = 0.5


res = 0.03
pixel_size = 0.025
idx = 14
det = 0
clock_spacing = 0.003 # ps

add_last_bin = 0.5

t_first = digitize_return_only_single_hits(pixel_size,x_list[idx][det],y_list[idx][det],t_list[idx][det])
t_all = t_list[idx][det]


t = smeared(t_first,res)
middle = np.round(np.mean(t), decimals=2)



for i in range(3):

    t = smeared(t_first,res)
    middle = np.round(np.mean(t), decimals=1)

    n_clocks = 32*(2**i)
    
    add_to_end = 0.5

    middle = np.round(np.mean(t), decimals=1)

    min_t = middle - (n_clocks/2)*clock_spacing
    regular_max_t = min_t + (n_clocks-1)*clock_spacing

    overflow_max_t = regular_max_t + clock_spacing + add_to_end

    my_edges = np.concatenate([np.arange(min_t,regular_max_t + clock_spacing,clock_spacing),[overflow_max_t]])
    counts, edges = np.histogram(t, bins=my_edges)


    # Add all t < min_t to the first bin
    counts[0] += np.sum(t < min_t)

    # Plot cumulative histogram
    cumulative = np.cumsum(counts)

    # Draw actual filled histogram bars
    widths = np.diff(edges)

    ax.bar(
        edges[:-1],
        cumulative,
        width=widths,
        align='edge',
        zorder=10-i,
        alpha=0.85,
        label=f'{n_clocks} clocks'
    )
    
    threshold = 0.5
    timing = get_time_at_threshold(t,threshold,n_clocks,clock_spacing,add_to_end)
    target = threshold*cumulative[-1]
    print(target,cumulative[-1])
    if(i==0):
        ax.axvline(timing,color='red',zorder=14,label=f'time at threshold: {timing:.3f}ns')
        ax.axhline(target,color='firebrick',zorder=14,label=f'threshold ({threshold*100:.0f}% of last bin)')
    
fs = 18

ax.grid(alpha=0.5)

ax.set_ylabel('number of hits')
ax.set_xlabel('time [ns]')
legend = ax.legend(framealpha=0.95)
legend.set_zorder(100)

# plt.xticks(np.arange(min_t, max_t, res))
# ax.tick_params(axis='x', which='major', labelsize=5)

# ax.set_xlim(241.2, 242.5)
plt.title(f'Timing histogram, {(clock_spacing*1000):.0f}ps bins, clocked around mean time + {add_to_end*1000:.0f}ps overflow bin\nonly single-hit {pixel_size*1000:.0f}um pixels, gap {idx} (cell {cell_list[idx]}), sensor {det}')
plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
pixel_size = 0.025
threshold = 0.5

n_clocks = 32
clock_spacings = [0.003,0.006]
add_to_end = 0.5

calc_clocks = n_clocks

print(f'digitized sensor: {pixel_size*1000:.0f}μm pixels')
print(f'threshold: {threshold*100:.0f}% of max.')
print(f'number of clocks: {n_clocks}')


t_difs_single = []


for i,spacing in enumerate(clock_spacings):

    print(f'clock spacing: {spacing*1000:.0f}ps')

    t_difs_single.append([])
    
    for idx in np.arange(0,24):
        
        threshold_times = []
    
        for det in np.arange(0,2*n_layers):
    
            t_first = digitize_return_only_single_hits(pixel_size,x_list[idx][det],y_list[idx][det],t_list[idx][det])

            timing = get_time_at_threshold(t_first,threshold,n_clocks,spacing,add_to_end)
    
            threshold_times.append(timing)
            
    
        t_difs_single[i].append(np.mean(threshold_times[n_layers:]) - np.mean(threshold_times[:n_layers]))
    
        print(f'gap {idx}, t difs (single hits):  {t_difs_single[i][idx]:.4f} ns')
        
       

In [ ]:
pixel_size = 0.025
threshold = 0.5

n_clocks = 32
clock_spacings_1plane = [0.003,0.006]
add_to_end = 1

calc_clocks_1plane = n_clocks

print(f'digitized sensor: {pixel_size*1000:.0f}μm pixels')
print(f'threshold: {threshold*100:.0f}% of max.')
print(f'number of clocks: {n_clocks}')


t_difs_single_1plane = []


for i,spacing in enumerate(clock_spacings):

    print(f'clock spacing: {spacing*1000:.0f}ps')

    t_difs_single_1plane.append([])
    
    for idx in np.arange(0,24):
        
        threshold_times = []
    
        for det in [0,3]:
    
            t_first = digitize_return_only_single_hits(pixel_size,x_list[idx][det],y_list[idx][det],t_list[idx][det])

            timing = get_time_at_threshold(t_first,threshold,n_clocks,spacing,add_to_end)
    
            threshold_times.append(timing)
            
    
        t_difs_single_1plane[i].append(threshold_times[1] - threshold_times[0])
    
        print(f'gap {idx}, t difs (single hits):  {t_difs_single_1plane[i][idx]:.4f} ns')
        
       

In [ ]:
pixel_size = 0.025
threshold = 0.5

n_clocks = 32
clock_spacings = [0.003,0.006]
add_to_end = 0.5

calc_clocks_approxmid = n_clocks

print(f'digitized sensor: {pixel_size*1000:.0f}μm pixels')
print(f'threshold: {threshold*100:.0f}% of max.')
print(f'number of clocks: {n_clocks}')


t_difs_single_approxmid = []
mid_dec = 2

for i,spacing in enumerate(clock_spacings):

    print(f'clock spacing: {spacing*1000:.0f}ps')

    t_difs_single_approxmid.append([])
    
    for idx in np.arange(0,24):
        
        threshold_times = []
    
        for det in [0,3]:
    
            t_first = digitize_return_only_single_hits(pixel_size,x_list[idx][det],y_list[idx][det],t_list[idx][det])

            timing = get_time_at_threshold_approxmid(t_first,threshold,n_clocks,spacing,add_to_end,mid_dec)
    
            threshold_times.append(timing)
            
    
        t_difs_single_approxmid[i].append(threshold_times[1] - threshold_times[0])
    
        print(f'gap {idx}, t difs (single hits):  {t_difs_single_approxmid[i][idx]:.4f} ps')
        
       

In [ ]:
def digitize_return_all_hits_boundary(pixel_size_in, boundary_in,
                                      x_in, y_in, t_in):

    size_x = 81.5

    pixel_size = pixel_size_in
    boundary = boundary_in
    pitch = pixel_size + boundary
    

    x_in = np.asarray(x_in)
    y_in = np.asarray(y_in)
    t_in = np.asarray(t_in)

    # Number of complete pixel pitches
    n_pixels = int(np.floor((2 * size_x) / pitch))

    # Position relative to detector lower-left corner
    x_rel = x_in + size_x
    y_rel = y_in + size_x

    # Pixel/pitch index
    ix = np.floor(x_rel / pitch).astype(int)
    iy = np.floor(y_rel / pitch).astype(int)

    # Position within each 30 um pitch
    x_local = x_rel - ix * pitch
    y_local = y_rel - iy * pitch

    # Hit is accepted only if BOTH x and y are
    # inside the 25 um active area
    valid = (
        (ix >= 0) & (ix < n_pixels) &
        (iy >= 0) & (iy < n_pixels) &
        (x_local >= 0) & (x_local < pixel_size) &
        (y_local >= 0) & (y_local < pixel_size)
    )

    return (
        x_in[valid],
        y_in[valid],
        t_in[valid],
        valid
    )


def digitize_return_first_hits_boundary_pos(pixel_size_in, boundary_in,
                                      x_in, y_in, t_in):

    size_x = 81.5

    pixel_size = pixel_size_in
    boundary = boundary_in
    pitch = pixel_size + boundary
    bins_x = int(2*size_x / pixel_size_in)

    x_in = np.asarray(x_in)
    y_in = np.asarray(y_in)
    t_in = np.asarray(t_in)

    # Number of complete pixel pitches
    n_pixels = int(np.floor((2 * size_x) / pitch))

    # Position relative to detector lower-left corner
    x_rel = x_in + size_x
    y_rel = y_in + size_x

    # Pixel/pitch index
    ix = np.floor(x_rel / pitch).astype(int)
    iy = np.floor(y_rel / pitch).astype(int)

    # Position within each 30 um pitch
    x_local = x_rel - ix * pitch
    y_local = y_rel - iy * pitch

    # Hit is accepted only if BOTH x and y are
    # inside the 25 um active area
    valid = (
        (ix >= 0) & (ix < n_pixels) &
        (iy >= 0) & (iy < n_pixels) &
        (x_local >= 0) & (x_local < pixel_size) &
        (y_local >= 0) & (y_local < pixel_size)
    )

    ix, iy, x_in, y_in, t_in = ix[valid], iy[valid],  x_in[valid], y_in[valid], t_in[valid]
    
    
    pixel_id = ix * bins_x + iy

    
    print(len(pixel_id))
    
    order = np.argsort(pixel_id)
    pixel_id_sorted = pixel_id[order]
    t_sorted = t_in[order]
    x_sorted = x_in[order]
    y_sorted = y_in[order]

    

    _, first_indices = np.unique(pixel_id_sorted, return_index=True)
    t_first_hits = t_sorted[first_indices]
    x_first_hits = x_sorted[first_indices]
    y_first_hits = y_sorted[first_indices]

    return x_first_hits,y_first_hits,t_first_hits

def digitize_return_single_hits_boundary_pos(pixel_size_in, boundary_in,
                                      x_in, y_in, t_in):

    size_x = 81.5

    pixel_size = pixel_size_in
    boundary = boundary_in
    pitch = pixel_size + boundary
    bins_x = int(2*size_x / pixel_size_in)

    x_in = np.asarray(x_in)
    y_in = np.asarray(y_in)
    t_in = np.asarray(t_in)

    # Number of complete pixel pitches
    n_pixels = int(np.floor((2 * size_x) / pitch))

    # Position relative to detector lower-left corner
    x_rel = x_in + size_x
    y_rel = y_in + size_x

    # Pixel/pitch index
    ix = np.floor(x_rel / pitch).astype(int)
    iy = np.floor(y_rel / pitch).astype(int)

    # Position within each 30 um pitch
    x_local = x_rel - ix * pitch
    y_local = y_rel - iy * pitch

    # Hit is accepted only if BOTH x and y are
    # inside the 25 um active area
    valid = (
        (ix >= 0) & (ix < n_pixels) &
        (iy >= 0) & (iy < n_pixels) &
        (x_local >= 0) & (x_local < pixel_size) &
        (y_local >= 0) & (y_local < pixel_size)
    )

    ix, iy, x_in, y_in, t_in = ix[valid], iy[valid],  x_in[valid], y_in[valid], t_in[valid]

    
    pixel_id = ix * bins_x + iy

    
    order = np.argsort(pixel_id)
    pixel_id_sorted = pixel_id[order]
    t_sorted = t_in[order]
    x_sorted = x_in[order]
    y_sorted = y_in[order]

    unique_ids, first_indices, counts = np.unique(
        pixel_id_sorted, return_index=True, return_counts=True
    )
    
    # Keep only pixels that were hit exactly once
    single_hit_mask = counts == 1
    single_hit_indices = first_indices[single_hit_mask]
    
    t_single_hits = t_sorted[single_hit_indices]

    x_single_hits = x_sorted[single_hit_indices]
    y_single_hits = y_sorted[single_hit_indices]

    return x_single_hits, y_single_hits, t_single_hits

def digitize_return_single_hits_boundary(pixel_size_in, boundary_in,
                                      x_in, y_in, t_in):

    size_x = 81.5

    pixel_size = pixel_size_in
    boundary = boundary_in
    pitch = pixel_size + boundary
    bins_x = int(2*size_x / pixel_size_in)

    x_in = np.asarray(x_in)
    y_in = np.asarray(y_in)
    t_in = np.asarray(t_in)

    # Number of complete pixel pitches
    n_pixels = int(np.floor((2 * size_x) / pitch))

    # Position relative to detector lower-left corner
    x_rel = x_in + size_x
    y_rel = y_in + size_x

    # Pixel/pitch index
    ix = np.floor(x_rel / pitch).astype(int)
    iy = np.floor(y_rel / pitch).astype(int)

    # Position within each 30 um pitch
    x_local = x_rel - ix * pitch
    y_local = y_rel - iy * pitch

    # Hit is accepted only if BOTH x and y are
    # inside the 25 um active area
    valid = (
        (ix >= 0) & (ix < n_pixels) &
        (iy >= 0) & (iy < n_pixels) &
        (x_local >= 0) & (x_local < pixel_size) &
        (y_local >= 0) & (y_local < pixel_size)
    )

    ix, iy, x_in, y_in, t_in = ix[valid], iy[valid],  x_in[valid], y_in[valid], t_in[valid]

    
    pixel_id = ix * bins_x + iy

    
    order = np.argsort(pixel_id)
    pixel_id_sorted = pixel_id[order]
    t_sorted = t_in[order]
    x_sorted = x_in[order]
    y_sorted = y_in[order]

    unique_ids, first_indices, counts = np.unique(
        pixel_id_sorted, return_index=True, return_counts=True
    )
    
    # Keep only pixels that were hit exactly once
    single_hit_mask = counts == 1
    single_hit_indices = first_indices[single_hit_mask]
    
    t_single_hits = t_sorted[single_hit_indices]

    x_single_hits = x_sorted[single_hit_indices]
    y_single_hits = y_sorted[single_hit_indices]

    
    return t_single_hits

In [ ]:
idx = 12
det = 0

pixel_size = 0.025       # mm = 25 um
boundary = 0.03       # mm = 5 um
pitch = pixel_size + boundary

size_x = 81.5

x = np.asarray(x_list[idx][det])
y = np.asarray(y_list[idx][det])
t = np.asarray(t_list[idx][det])


x_dig, y_dig, t_dig, valid = digitize_return_all_hits_boundary(pixel_size,boundary,x,y,t)

x_first, y_first, t_first = digitize_return_first_hits_boundary_pos(pixel_size,boundary,x,y,t)

x_single, y_single, t_single, =digitize_return_single_hits_boundary_pos(pixel_size,boundary,x,y,t)

# Rejected hits
x_rejected = x[~valid]
y_rejected = y[~valid]
t_rejected = t[~valid]

print(f"Original hits:   {len(x)}")
print(f"Accepted hits:   {len(x_dig)}")
print(f"Rejected hits:   {len(x_rejected)}")
print(f"First accepted hits: {len(x_first)}")
print(f"Single accepted hits: {len(x_single)}")


zoom_size = 0.25   # mm

xmin = -zoom_size / 2
xmax =  zoom_size / 2
ymin = -zoom_size / 2
ymax =  zoom_size / 2


fig, ax = plt.subplots(figsize=(8, 8))



# draw pixels

ix_start = int(np.floor((xmin + size_x) / pitch))
ix_end = int(np.ceil((xmax + size_x) / pitch))

iy_start = int(np.floor((ymin + size_x) / pitch))
iy_end = int(np.ceil((ymax + size_x) / pitch))


for ix_global in range(ix_start, ix_end):

    for iy_global in range(iy_start, iy_end):

        x0 = -size_x + ix_global * pitch
        y0 = -size_x + iy_global * pitch

        # Draw ONLY the 25 um active region
        ax.pcolormesh(
            [x0, x0 + pixel_size],
            [y0, y0 + pixel_size],
            np.ones((1, 1)),
            cmap='YlGnBu',
            vmin=0,
            vmax=1,
            shading="flat",
            alpha=0.35
        )



accepted_zoom = ((x_dig >= xmin) & (x_dig <= xmax) & (y_dig >= ymin) & (y_dig <= ymax))

rejected_zoom = ((x_rejected >= xmin) & (x_rejected <= xmax) & (y_rejected >= ymin) & (y_rejected <= ymax))

first_zoom = ((x_first >= xmin) & (x_first <= xmax) & (y_first >= ymin) & (y_first <= ymax))

single_zoom = ((x_single >= xmin) & (x_single <= xmax) & (y_single >= ymin) & (y_single <= ymax))


ax.scatter(
    x_dig[accepted_zoom],
    y_dig[accepted_zoom],
    color="red",
    s=20,
    zorder=10,
    label=f"Hits on pixel ({len(x_dig)/len(x) * 100:.0f}%)"
)

ax.scatter(
    x_rejected[rejected_zoom],
    y_rejected[rejected_zoom],
    color="#1C1C1C",
    s=20,
    zorder=11,
    label=f"Rejected hits on boundary ({len(x_rejected)/len(x) * 100:.0f}%)"
)

ax.scatter(
    x_first[first_zoom],
    y_first[first_zoom],
    color="yellow",
    s=10,
    zorder=12,
    label=f"First hits on pixel ({len(x_first)/len(x) * 100:.0f}%)"
)

ax.scatter(
    x_single[single_zoom],
    y_single[single_zoom],
    color="blue",
    s=50,
    zorder=15,
    label=f"Single hits on pixel ({len(x_single)/len(x) * 100:.0f}%)"
)


ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_aspect("equal")

ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")

ax.set_title(
    f"Detector geometry\n"
    f"Pixel = {pixel_size * 1000:.0f} µm, "
    f"Boundary = {boundary * 1000:.0f} µm, "
    f"Pitch = {pitch * 1000:.0f} µm"
,fontsize=14)

legend = ax.legend(loc="upper right",fontsize=14)
legend.set_zorder(100)
legend.get_frame().set_alpha(0.9)

plt.tight_layout()
plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
plt.rcParams['grid.alpha'] = 0.5


res = 0.03
pixel_size = 0.025
boundary = 0.005
idx = 14
det = 0
clock_spacing = 0.003 # ps

add_last_bin = 0.5

t_first = digitize_return_only_single_hits(pixel_size,x_list[idx][det],y_list[idx][det],t_list[idx][det])
t_all = t_list[idx][det]


t = smeared(t_first,res)
middle = np.round(np.mean(t), decimals=2)


for i in range(3):

    t = smeared(t_first,res)
    middle = np.round(np.mean(t), decimals=1)

    n_clocks = 32*(2**i)
    
    add_to_end = 0.5

    middle = np.round(np.mean(t), decimals=1)

    min_t = middle - (n_clocks/2)*clock_spacing
    regular_max_t = min_t + (n_clocks-1)*clock_spacing

    overflow_max_t = regular_max_t + clock_spacing + add_to_end

    my_edges = np.concatenate([np.arange(min_t,regular_max_t + clock_spacing,clock_spacing),[overflow_max_t]])
    counts, edges = np.histogram(t, bins=my_edges)


    # Add all t < min_t to the first bin
    counts[0] += np.sum(t < min_t)

    # Plot cumulative histogram
    cumulative = np.cumsum(counts)

    # Draw actual filled histogram bars
    widths = np.diff(edges)

    ax.bar(
        edges[:-1],
        cumulative,
        width=widths,
        align='edge',
        zorder=10-i,
        alpha=0.85,
        label=f'{n_clocks} clocks'
    )
    
    threshold = 0.5
    timing = get_time_at_threshold(t,threshold,n_clocks,clock_spacing,add_to_end)
    target = threshold*cumulative[-1]
    print(target,cumulative[-1])
    if(i==0):
        ax.axvline(timing,color='red',zorder=14,label=f'time at threshold: {timing:.3f}ns')
        ax.axhline(target,color='firebrick',zorder=14,label=f'threshold ({threshold*100:.0f}% of last bin)')
    
fs = 18

ax.grid(alpha=0.5)

ax.set_ylabel('number of hits')
ax.set_xlabel('time [ns]')
legend = ax.legend(framealpha=0.95)
legend.set_zorder(100)

# plt.xticks(np.arange(min_t, max_t, res))
# ax.tick_params(axis='x', which='major', labelsize=5)

# ax.set_xlim(241.2, 242.5)
plt.title(f'Timing histogram, {(clock_spacing*1000):.0f}ps bins, clocked around mean time + {add_to_end*1000:.0f}ps overflow bin\nonly single-hit {pixel_size*1000:.0f}um pixels, gap {idx}, sensor {det}')
plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
n_clocks = 32
spacing  = 0.003
add_to_end = 0.5

boundary_spacing = spacing
t_difs_single_small_boundary = []
t_difs_single_big_boundary = []

pixel_size = 0.025
small_boundary = 0.005
big_boundary = 0.03

small_pitch = pixel_size + small_boundary
big_pitch = pixel_size + big_boundary


print(f'digitized sensor: {pixel_size*1000:.0f}μm pixels')


print(f'clock spacing: {spacing*1000:.0f}ps')
print(f'single planes!')

for idx in np.arange(0,24):
    
    threshold_times_small = []
    threshold_times_big = []

    for det in [0,3]:

        t_small = digitize_return_single_hits_boundary(pixel_size,small_boundary,x_list[idx][det],y_list[idx][det],t_list[idx][det])
        t_big = digitize_return_single_hits_boundary(pixel_size,big_boundary,x_list[idx][det],y_list[idx][det],t_list[idx][det])

        timing_small = get_time_at_threshold(t_small,threshold,n_clocks,spacing,add_to_end)
        timing_big = get_time_at_threshold(t_big,threshold,n_clocks,spacing,add_to_end)

        threshold_times_small.append(timing_small)
        threshold_times_big.append(timing_big)
        

    t_difs_single_small_boundary.append(threshold_times_small[1] - threshold_times_small[0])
    t_difs_single_big_boundary.append(threshold_times_big[1] - threshold_times_big[0])

    print(f'gap {idx}, all single hits: t difs ({small_boundary*1000:.0f}um boundary):  {t_difs_single_small_boundary[idx]:.4f} ns, t difs ({big_boundary*1000:.0f}um boundary): {t_difs_single_big_boundary[idx]:.4f} ns')
        

       

In [ ]:
target = 0.5*cumulative[-1]

index, value = min(
        enumerate(cumulative),
        key=lambda item: abs(item[1] - target)
    )

print(index,value,target,cumulative[-1])

In [ ]:
min_t+index*clock_spacing


In [ ]:
from collections import Counter
from scipy.stats import landau, norm, trim_mean, trimboth, trim1


def event_time_dict(event_ids,times):
    counts = Counter(event_ids)
    return {e: t for e, t in zip(event_ids, times) if counts[e] == 1}


true_t_difs = []

true_t_difs_all = []
# 1e6




for idx in np.arange(0,24):
 
    det0 = event_time_dict(event_list[idx][0], t_list[idx][0])
    det1 = event_time_dict(event_list[idx][1], t_list[idx][1])
    det2 = event_time_dict(event_list[idx][2], t_list[idx][2])
    det3 = event_time_dict(event_list[idx][3], t_list[idx][3])
    det4 = event_time_dict(event_list[idx][4], t_list[idx][4])
    det5 = event_time_dict(event_list[idx][5], t_list[idx][5])

    common = det0.keys() & det1.keys() & det2.keys() & det3.keys() & det4.keys() & det5.keys()

    dt = [np.mean([det3[e], det4[e], det5[e]]) - np.mean([det0[e], det1[e], det2[e]]) for e in common]
            
    trim_frac = 0.001   # remove 0.1% from right tail (default for trim1 is tail='right')

    t_trim = trim1(dt,trim_frac)
    
    # t_trim = dt

    true_t_difs_all.append(t_trim)
    
    t_mean = np.mean(t_trim)
    t_spread = 5*np.std(t_trim)
    nbins=int(np.round(2*t_spread/0.005)) # 5ps bins

    mu, sigma, A, mu_uncertainty, bin_centers, hist_counts = get_gaussian_params(t_trim, nbins)
    true_t_difs.append(mu)
        
    print(f'gap {idx}: mean TRUE t difs: {true_t_difs[idx]:.4f}')

print("done") 
   

In [ ]:
def get_true_p(idx,group):

    px = np.concatenate([px_list[idx][group*3], px_list[idx][group*3+1], px_list[idx][group*3+2]])
    py = np.concatenate([py_list[idx][group*3], py_list[idx][group*3+1], py_list[idx][group*3+2]])
    pz = np.concatenate([pz_list[idx][group*3], pz_list[idx][group*3+1], pz_list[idx][group*3+2]])
        
    p_placehold = []

    for k in np.arange(0,len(px)):
        p_placehold.append(np.sqrt(px[k]**2 + py[k]**2 + pz[k]**2))

    trim_frac = 0.001   # remove 5% from each tail

    p_trim = trimboth(p_placehold,trim_frac)

    p_mean = np.mean(p_trim)
    p_spread = 5*np.std(p_trim)
    nbins=int(np.round(2*p_spread/0.03))

    lower_bounds_gauss = [1, 0.001, 1.0]
    upper_bounds_gauss = [1000, 1000.0, 4000000.0]


    hist_counts_func, bin_edges = np.histogram(p_trim,
        range=(p_mean - p_spread, p_mean + p_spread),
        bins=nbins
    )

    bin_centers_func = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    mu_init = bin_centers_func[np.argmax(hist_counts_func)]
    sigma_init = np.std(bin_centers_func)
    A_init = np.max(hist_counts_func)

    params_gauss, covariance_gauss = curve_fit(
        gaussian,
        bin_centers_func,
        hist_counts_func,
        p0=[mu_init, sigma_init, A_init],
        bounds=(lower_bounds_gauss,upper_bounds_gauss)
    )

    mu, sigma, A = params_gauss

    return mu

 

p_means = []

for idx in np.arange(0,24):
    
    means_temp = []

    for group in np.arange(0,2):

        mean = get_true_p(idx,group)
        
        means_temp.append(mean)
        

    p_means.append(np.mean(means_temp))
    print(f'gap {idx}: p mean: {p_means[idx]:.2f} MeV')
        

In [ ]:
def get_p(t_in, L_in):

    m_mu = 105.7 # MeV/c^2
    c = 299.792 # mm/ns

    beta = L_in/(c*t_in)
    gamma = 1/np.sqrt(1-beta**2)
    p = beta*gamma*m_mu

    return p

    
  
    # dt = clock_spacing
    # dT = np.sqrt(2) * dt # uncertainty on difference in timing between the two groups

    # dL = cell_trlen_stdev[idx]/np.sqrt(n_traces)
    # # print(dL)

    # beta = L/(c*t_difs_single[idx])
    # gamma = 1/np.sqrt(1-beta**2)

    # db = beta * np.sqrt((dL/L)**2 + (dT/np.mean(t_difs_single[idx]))**2)
    
    # dp = float(m_mu * gamma**3 * db)
    # p_err.append(dp)

    



In [ ]:
gap_zs = []

for idx in np.arange(0,24):
    gap_z_pos = (z_det_avgs[idx][0] + z_det_avgs[idx][1])/2
    gap_zs.append(gap_z_pos/1000)
    

reco_p = []

reco_p_true = []

p_err = []

reco_p_1plane = []

reco_p_approxmid = []

reco_p_smallboundary = []
reco_p_bigboundary = []



for i,spacing in enumerate(clock_spacings):

    reco_p.append([])

    for idx in np.arange(0,24):

        L = cell_trlen_avgs_path[idx]
    
        reco_p[i].append(get_p(t_difs_single[i][idx],L))

for i,spacing in enumerate(clock_spacings_1plane):

    reco_p_1plane.append([])
    reco_p_approxmid.append([])

    for idx in np.arange(0,24):

        L = cell_trlen_avgs_path[idx]
    
        reco_p_1plane[i].append(get_p(t_difs_single_1plane[i][idx],L))
        reco_p_approxmid[i].append(get_p(t_difs_single_approxmid[i][idx],L))

for idx in range(len(true_t_difs)):
    L = cell_trlen_avgs_path[idx]
    reco_p_true.append(get_p(true_t_difs[idx],L))
    reco_p_smallboundary.append(get_p(t_difs_single_small_boundary[idx],L))
    reco_p_bigboundary.append(get_p(t_difs_single_big_boundary[idx],L))

 
fig,ax=plt.subplots(figsize=(18,7))
plt.rcParams['grid.alpha'] = 0.5


print(f'\nmean track length average: {np.mean(cell_trlen_avgs_path):.2f} mm')


alpha=0.9

marker_size = 120


ax.fill_between(gap_zs,1.01*np.array(p_means),0.99*np.array(p_means),color="tab:orange",alpha=0.2,zorder=0,label="within 1% of true mean momentum")
ax.scatter(gap_zs, p_means,label=f'true mean momentum', 
           alpha=0.9, color="tab:orange",
           zorder=100,s=marker_size*0.8)

# ax.scatter(np.array(gap_zs), reco_p_true,   marker='^', 
#            label=f'mean of true particle transit times across gap, 1e6', color='fuchsia', s=marker_size)


# ax.errorbar(np.array(gap_zs), reco_p_single_6, p_err,  marker='^', 
#            label=f'reconstruction: {threshold*100:.0f}% threshold, {clock_spacing*1000:.0f}ps clocks', color='fuchsia')

# ax.scatter(np.array(gap_zs), reco_p[0],  marker='s',
#            label=f'reconstruction: MULTIPLE PLANES, {threshold*100:.0f}% threshold, {calc_clocks} clocks, {clock_spacings[0]*1000:.0f}ps spacing', 
#            color='dodgerblue',s=marker_size,alpha=alpha)

ax.scatter(np.array(gap_zs), reco_p_1plane[0],  marker='s',
           label=f'reconstruction: only single-hit {pixel_size*1000:.0f}μm pixels\n  {threshold*100:.0f}% threshold, {calc_clocks} clocks, {clock_spacings[0]*1000:.0f}ps spacing', 
           color='dodgerblue',s=marker_size,alpha=alpha)

# ax.scatter(np.array(gap_zs), reco_p[1],  marker='s', 
#            label=f'reconstruction: {threshold*100:.0f}% threshold, 32 clocks, {clock_spacings[1]*1000:.0f}ps clocks', 
#            color='fuchsia',s=marker_size,alpha=alpha)

# ax.scatter(np.array(gap_zs), reco_p[2],  marker='*', 
#            label=f'reconstruction: {threshold*100:.0f}% threshold, 32 clocks, {clock_spacings[2]*1000:.0f}ps clocks', 
#            color='tomato',s=marker_size,alpha=alpha)

# ax.scatter(np.array(gap_zs), reco_p[3],  marker='^', 
#            label=f'reconstruction: {threshold*100:.0f}% threshold, {calc_clocks} clocks, {clock_spacings[3]*1000:.0f}ps clocks', 
#            color='turquoise',s=marker_size,alpha=alpha)

# ax.scatter(np.array(gap_zs), reco_p_1plane[1],  marker='s', 
#            label=f'reconstruction:\n  only single-hit {pixel_size*1000:.0f}μm pixels\n  {threshold*100:.0f}% threshold, {calc_clocks_1plane} clocks, {clock_spacings_1plane[1]*1000:.0f}ps clock spacing', 
#            color='green',s=marker_size,alpha=alpha)

# ax.plot(np.array(gap_zs), reco_p_approxmid[1],  marker='s', 
#            label=f'reconstruction:\n  only single-hit pixels\n  {threshold*100:.0f}% threshold, {calc_clocks_1plane} clocks, {clock_spacings_1plane[1]*1000:.0f}ps clock spacing, approx. mean of clocks (1ps)', 
#            color='limegreen',alpha=alpha)

# ax.scatter(np.array(gap_zs), reco_p_approxmid[1],  marker='d', 
#            label=f'reconstruction:\n  only single-hit pixels\n  {threshold*100:.0f}% threshold, {calc_clocks_1plane} clocks, {clock_spacings_1plane[1]*1000:.0f}ps clock spacing, approx. mean of clocks ({10**(3-mid_dec):.0f}ps)', 
#            color='limegreen',s=marker_size,alpha=alpha)

# ax.scatter(np.array(gap_zs), reco_p_1plane[1],  marker='d', 
#            label=f'reconstruction: {threshold*100:.0f}% threshold, {calc_clocks} clocks, {clock_spacings[1]*1000:.0f}ps clocks', 
#            color='green',s=marker_size,alpha=alpha)

ax.scatter(np.array(gap_zs), reco_p_bigboundary,  marker='d', 
           label=f'reconstruction: only single-hit {pixel_size*1000:.0f}μm pixels, {big_boundary*1000:.0f}μm spacing\n  {threshold*100:.0f}% threshold, {n_clocks} clocks, {boundary_spacing*1000:.0f}ps clock spacing', 
           color='darkviolet',s=marker_size,alpha=alpha)


fs = 16

ax.grid(alpha=0.5)
ax.legend(loc='upper right',fontsize=fs)

ax.set_xlabel('z position [m]',fontsize=fs)
ax.set_ylabel('mu+ momentum [MeV/c]',fontsize=fs)
ax.tick_params(axis='both', which='major', labelsize=18)
# ax.set_ylim([205.5,214.25])
plt.title(f'Momentum, 200 MeV μ+ bunches, 60 cell cooling channel',fontsize=30)
plt.tight_layout(pad=2.0)
plt.savefig("/Users/cicihanna/Documents/mucol/my_papers/momentum.png")



In [ ]:
for i in range(len(p_means)):
    print(p_means[i],reco_p[0][i],gap_zs[i])

In [ ]:
len(reco_p[0])

In [ ]:
(2*81.5/0.025)**2 / 10**6
